# FourCastNeXt Training Demo



!Note! This model is not identical to the model from the paper. The goal is eventually to provide a version which reproduces the results of the paper, and also a modified version which can support some specific use cases.

Training can be initiated a few different ways, and there are good reasons for each use case. It can be done:

 1. Inside a Jupyter notebook, as Python code, with or without experiment tracking
 2. From the command-line or in a Jupyter notebook using the command-line execution magic, leveraging "Hydra" for experiment tracking
 3. Using a supercomputer job scheduler to submit training jobs as part of a queuing system

This notebook will start with the first approach to illuminate the process, but for those involved in research into new model archictures, (2) and (3) offer more flexibility for training multiple models at once, and for operating across multiple HPC nodes or cloud instances to accelerate training and discovery.

Performing the training from the command-line is basically a matter of making a .py version of this notebook. Job schedule submission is a matter of calling that .py file via a job script wrapper.

## Training from ERA5 - Limited Variable, Early Stopping Demonstration



In [1]:
import hydra
from omegaconf import OmegaConf
import site_archive_nci
import fourcastnext
import pathlib

In [2]:
config_dir = str((pathlib.Path(fourcastnext.__file__).parent / '../../Training/limited_variables_early_stopping').resolve())

In [3]:
initialsed = hydra.initialize_config_dir(version_base=None, config_dir=config_dir)
cfg = hydra.compose(config_name="limited_vars_early_stop.yaml")
# print(cfg)

In [4]:
import pyearthtools.training
import pyearthtools.pipeline

In [5]:
splits = {
    "train_split": pyearthtools.pipeline.iterators.DateRange(*cfg.data.splits.train),
    "valid_split": pyearthtools.pipeline.iterators.DateRange(*cfg.data.splits.valid),
}

In [6]:
pipeline_path = str((pathlib.Path(fourcastnext.__file__).parent / '../../Training/pipelines/early_stopping.pipe').resolve())

In [7]:
datamodule = pyearthtools.training.data.lightning.PipelineLightningDataModule(
    pipeline_path,  # type: ignore
    **splits,
    **cfg.data.module,
)

In [8]:
# datamodule.pipelines

In [9]:
# Uncomment this to see the output of the pipeline at a given hour
# datamodule.pipelines['2001-01-01T00']

In [10]:
model = hydra.utils.instantiate(cfg.model)
# This is the FourCastNext Lightning Model. Uncomment the line below to see
# a prinout of the layers in the model
# model

In [11]:
trainer = pyearthtools.training.lightning.Train(
    model,
    datamodule,
    path=cfg.path,
    trainer_kwargs={'num_sanity_val_steps': 0},
    **OmegaConf.to_object(cfg.trainer),  # type: ignore
)



In [12]:
# This takes 10 plus minutes just to get started, comprising
# - Sanity Checking
# - Sanity Checking Data Loader
# - Just sitting there for another 10+ minutes with no explanation

# Uncomment this line to perform fitting. Commented out to avoid incurring lengthy
# training by accident
# trainer.fit()